# Feature Engineering

## Objective

The objective of this notebook is to extract handcrafted image features that can distinguish genuine photographs from screen recapture images.

Instead of directly training deep learning models, several computer vision descriptors are computed to capture texture, color, frequency, edge, and reflection characteristics.

These engineered features are later used to train classical machine learning models such as Random Forest and XGBoost.

## Import Required Libraries

Import all libraries required for image processing, feature extraction, numerical computation, and data manipulation.

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

def extract_basic_features(image_path):

    img = cv2.imread(image_path)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape

    # Brightness
    brightness = np.mean(gray)

    # Contrast
    contrast = np.std(gray)

    # Laplacian Variance (Sharpness)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Canny Edge Density
    edges = cv2.Canny(gray, 100, 200)

    edge_density = np.sum(edges > 0) / (h * w)

    return {
        "width": w,
        "height": h,
        "brightness": brightness,
        "contrast": contrast,
        "laplacian": lap_var,
        "edge_density": edge_density
    }

## Define Dataset Paths

Specify the directory paths for the real photograph and screen recapture image datasets.

In [ ]:
REAL_DIR = "../dataset/real"
SCREEN_DIR = "../dataset/screen"

## Load Image File Paths

Collect all valid image filenames from both classes while filtering supported image formats.

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

real_images = [
    f for f in os.listdir(REAL_DIR)
    if f.lower().endswith(IMAGE_EXTENSIONS)
]

screen_images = [
    f for f in os.listdir(SCREEN_DIR)
    if f.lower().endswith(IMAGE_EXTENSIONS)
]

print(f"Real Images   : {len(real_images)}")
print(f"Screen Images : {len(screen_images)}")

## Extract Basic Image Statistics

Iterate through every image and compute simple descriptive statistics that provide an initial understanding of both image classes.

In [ ]:
records = []

for folder, label in [
    (REAL_DIR, "real"),
    (SCREEN_DIR, "screen")
]:

    for file in os.listdir(folder):

        if not file.lower().endswith(IMAGE_EXTENSIONS):
            continue

        path = os.path.join(folder, file)

        features = extract_basic_features(path)

        features["label"] = label
        features["filename"] = file

        records.append(features)

df = pd.DataFrame(records)

df.head()

## Compare Class-wise Statistics

Calculate the average feature values for both classes to identify potential differences between genuine photographs and screen recapture images.

In [ ]:
df.groupby("label").mean(numeric_only=True)

## Frequency Domain Visualization

Visualize the Fourier Transform of sample images.

Screen recaptures often exhibit repetitive frequency patterns caused by display pixels and moiré artifacts that become visible in the frequency domain.

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    df[df.label=="real"]["brightness"],
    bins=20,
    alpha=0.6,
    label="Real"
)

plt.hist(
    df[df.label=="screen"]["brightness"],
    bins=20,
    alpha=0.6,
    label="Screen"
)

plt.xlabel("Brightness")
plt.ylabel("Count")
plt.legend()
plt.show()

## Fourier Transform Function

Define a reusable function for computing and visualizing the Fast Fourier Transform (FFT) of an image.

In [ ]:
def show_fft(image_path):

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # FFT
    fft = np.fft.fft2(img)

    fft_shift = np.fft.fftshift(fft)

    magnitude = np.log(np.abs(fft_shift) + 1)

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.imshow(img, cmap="gray")
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(magnitude, cmap="gray")
    plt.title("FFT Magnitude Spectrum")
    plt.axis("off")

    plt.show()

## FFT Example: Image Pair 1

Compare the frequency spectrum of one genuine photograph with one screen recapture image.

In [ ]:
show_fft("../dataset/real/image_1.jpeg")
show_fft("../dataset/screen/image_1.jpeg")

## FFT Example: Image Pair 2

Repeat the frequency-domain comparison using another pair of images to observe consistent patterns.

In [ ]:
show_fft("../dataset/real/image_2.jpeg")
show_fft("../dataset/screen/image_2.jpeg")

## FFT Example: Image Pair 3

Visualize additional samples to verify that the extracted frequency characteristics remain consistent across different images.

In [ ]:
show_fft("../dataset/real/image_3.jpeg")
show_fft("../dataset/screen/image_3.jpeg")

## Extract FFT Features

Define a function that computes numerical FFT-based descriptors for every image.

These descriptors summarize the frequency information and are later used as machine learning features.

In [ ]:
def extract_fft_features(image_path):

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Resize so every FFT is comparable
    img = cv2.resize(img, (256, 256))

    # FFT
    fft = np.fft.fft2(img)
    fft_shift = np.fft.fftshift(fft)

    magnitude = np.log(np.abs(fft_shift) + 1)

    h, w = magnitude.shape
    cy, cx = h // 2, w // 2

    # ---------- Overall statistics ----------

    fft_mean = np.mean(magnitude)
    fft_std = np.std(magnitude)

    # ---------- Low-frequency energy ----------

    center = magnitude[cy-20:cy+20, cx-20:cx+20]
    low_energy = np.mean(center)

    # ---------- High-frequency energy ----------

    mask = np.ones_like(magnitude, dtype=bool)
    mask[cy-20:cy+20, cx-20:cx+20] = False

    high_energy = np.mean(magnitude[mask])

    # Ratio
    hf_ratio = high_energy / (low_energy + 1e-8)

    # ---------- Horizontal line energy ----------

    horizontal = np.mean(magnitude[cy-2:cy+2, :])

    # ---------- Vertical line energy ----------

    vertical = np.mean(magnitude[:, cx-2:cx+2])

    return {
        "fft_mean": fft_mean,
        "fft_std": fft_std,
        "low_energy": low_energy,
        "high_energy": high_energy,
        "hf_ratio": hf_ratio,
        "horizontal_energy": horizontal,
        "vertical_energy": vertical
    }

## FFT Feature Example

Apply the FFT feature extraction function to a sample image and inspect the generated feature vector.

In [ ]:
extract_fft_features("../dataset/real/image_1.jpeg")

## Generate FFT Feature Dataset

Extract FFT descriptors for every image in the dataset and store them in a structured DataFrame.

In [ ]:
DATASET_PATH = "../dataset"

data = []

for label in ["real", "screen"]:

    folder = os.path.join(DATASET_PATH, label)

    for filename in os.listdir(folder):

        if filename.startswith("."):
            continue

        image_path = os.path.join(folder, filename)

        img = cv2.imread(image_path)

        height, width = img.shape[:2]

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        brightness = np.mean(gray)

        contrast = np.std(gray)

        laplacian = cv2.Laplacian(gray, cv2.CV_64F).var()

        edges = cv2.Canny(gray, 100, 200)

        edge_density = np.sum(edges > 0) / edges.size

        # FFT Features
        fft_features = extract_fft_features(image_path)

        features = {
            "width": width,
            "height": height,
            "brightness": brightness,
            "contrast": contrast,
            "laplacian": laplacian,
            "edge_density": edge_density,

            **fft_features,

            "label": label,
            "filename": filename
        }

        data.append(features)

df = pd.DataFrame(data)

df.head()

## Inspect FFT Features

Display the extracted FFT feature columns to verify that feature extraction was successful.

In [ ]:
fft_cols = [
    "fft_mean",
    "fft_std",
    "low_energy",
    "high_energy",
    "hf_ratio",
    "horizontal_energy",
    "vertical_energy"
]

df.groupby("label")[fft_cols].mean()

In [ ]:
df.to_csv("image_features_fft.csv", index=False)

In [ ]:
df.groupby("label")[fft_cols].mean()

## Local Binary Pattern (LBP)

Define a function for extracting Local Binary Pattern descriptors.

LBP captures local texture information and is widely used for texture classification tasks.

In [ ]:
from skimage.feature import local_binary_pattern

def extract_lbp_features(image_path):

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    img = cv2.resize(img, (256,256))

    radius = 3
    n_points = radius * 8

    lbp = local_binary_pattern(
        img,
        n_points,
        radius,
        method="uniform"
    )

    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_points + 3),
        range=(0, n_points + 2)
    )

    hist = hist.astype("float")
    hist /= hist.sum()

    return hist

In [ ]:
hist = extract_lbp_features("../dataset/real/image_1.jpeg")

print(len(hist))
print(hist)

## Color Feature Extraction

Extract color statistics such as mean, standard deviation, and channel-wise information.

Screen recapture images often exhibit different color distributions due to display illumination.

In [ ]:
def extract_color_features(image_path):

    img = cv2.imread(image_path)

    img = cv2.resize(img, (256,256))

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    rgb_mean = np.mean(img, axis=(0,1))
    rgb_std = np.std(img, axis=(0,1))

    hsv_mean = np.mean(hsv, axis=(0,1))
    hsv_std = np.std(hsv, axis=(0,1))

    return {

        "r_mean": rgb_mean[2],
        "g_mean": rgb_mean[1],
        "b_mean": rgb_mean[0],

        "r_std": rgb_std[2],
        "g_std": rgb_std[1],
        "b_std": rgb_std[0],

        "h_mean": hsv_mean[0],
        "s_mean": hsv_mean[1],
        "v_mean": hsv_mean[2],

        "h_std": hsv_std[0],
        "s_std": hsv_std[1],
        "v_std": hsv_std[2],
    }

In [ ]:
color_features = extract_color_features(image_path)

print(color_features)

## Histogram of Oriented Gradients (HOG)

Define a function for extracting HOG descriptors.

HOG captures edge orientation information that can reveal structural differences between genuine photographs and screen recapture images.

In [ ]:
from skimage.feature import hog

def extract_hog_features(image_path):

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    img = cv2.resize(img,(128,128))

    features = hog(

        img,

        orientations=9,

        pixels_per_cell=(8,8),

        cells_per_block=(2,2),

        feature_vector=True

    )

    return {

        "hog_mean": np.mean(features),

        "hog_std": np.std(features),

        "hog_max": np.max(features),

        "hog_min": np.min(features)

    }

In [ ]:
hog_features = extract_hog_features(image_path)

print(hog_features)

## Reflection Feature Extraction

Estimate reflection intensity present within an image.

Display reflections are frequently observed in photographs captured from electronic screens.

In [ ]:
def extract_reflection_score(image_path):

    img = cv2.imread(image_path)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    value = hsv[:,:,2]

    bright_pixels = np.sum(value > 240)

    score = bright_pixels / value.size

    return {

        "reflection_score": score

    }

In [ ]:
reflection = extract_reflection_score(image_path)

print(reflection)

## Moiré Pattern Detection

Define a feature for estimating moiré artifacts.

Moiré patterns are one of the most distinctive characteristics of screen recapture images.

In [ ]:
def extract_moire_score(image_path):

    gray = cv2.imread(image_path,0)

    gray = cv2.resize(gray,(256,256))

    lap = cv2.Laplacian(gray,cv2.CV_64F)

    score = np.var(lap)

    return {

        "moire_score": score

    }

In [ ]:
moire = extract_moire_score(image_path)

print(moire)

## Generate Final Feature Dataset

Combine all extracted descriptors into a single feature table containing every engineered feature for every image.

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

records = []

for folder, label in [
    (REAL_DIR, "real"),
    (SCREEN_DIR, "screen")
]:

    for filename in os.listdir(folder):

        if not filename.lower().endswith(IMAGE_EXTENSIONS):
            continue

        image_path = os.path.join(folder, filename)

        img = cv2.imread(image_path)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        h, w = gray.shape

        brightness = np.mean(gray)

        contrast = np.std(gray)

        laplacian = cv2.Laplacian(gray, cv2.CV_64F).var()

        edges = cv2.Canny(gray,100,200)

        edge_density = np.sum(edges>0)/(h*w)


        fft_features = extract_fft_features(image_path)

        color_features = extract_color_features(image_path)


        hog_features = extract_hog_features(image_path)

        reflection = extract_reflection_score(image_path)

        moire = extract_moire_score(image_path)

        lbp = extract_lbp_features(image_path)

        lbp_dict = {
            f"lbp_{i}": value
            for i, value in enumerate(lbp)
        }

        record = {

            "width": w,
            "height": h,

            "brightness": brightness,
            "contrast": contrast,

            "laplacian": laplacian,

            "edge_density": edge_density,

            **fft_features,

            **color_features,

            **hog_features,

            **reflection,

            **moire,

            **lbp_dict,

            "label": label,

            "filename": filename

        }

        records.append(record)

df = pd.DataFrame(records)

In [ ]:
df.head()

In [ ]:
print(df.shape)

In [ ]:
print(df.columns.tolist())

In [ ]:
df.to_csv("../engineered_features.csv", index=False)

# Summary

This notebook constructed a comprehensive handcrafted feature representation for every image in the dataset.

### Features extracted

- Fast Fourier Transform (FFT)
- Local Binary Patterns (LBP)
- Histogram of Oriented Gradients (HOG)
- Color Statistics
- Reflection Score
- Moiré Score
- Texture Descriptors

These features were consolidated into a structured dataset and exported as `engineered_features.csv`.

The generated feature set serves as the input for the classical machine learning models developed in the next notebook, where Random Forest and XGBoost classifiers are trained and evaluated.